# Week 4: Risk-Adjusted Returns
In Week 1 you compared raw return and volatility. That comparison isn't fair: an asset that takes twice the risk *should* earn more. This week you put both assets on equal footing and answer two questions:

1. **Per unit of risk, which one pays better?** (Sharpe and Sortino ratios)
2. **How much of each one is just the market?** (beta, alpha and R² from a regression on SPY)

"The market" is SPY and the risk-free rate is the 3-month T-bill. Both were saved in **step 4 of the setup notebook**. If you haven't run it since it was updated, do that first.

Work top to bottom. Each step says what to do, why, and what to look for. Cells with `___` are yours to fill in.

## Step 1: Load prices
Same as every week, plus SPY and the risk-free rate. Change `A` and `B` to your pair.

`.dropna()` keeps only the days where all three traded. If one of your assets is crypto, this drops weekends, so everything this week uses **252** days a year, not 365. If SPY is one of your assets, that's fine. Its beta will just come out as exactly 1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

A, B = "NVDA", "DUK"   # <- your pair

def load(ticker):
    return pd.read_csv(f"../data/{ticker}.csv", index_col=0, parse_dates=True)["Adj Close"]

prices = pd.DataFrame({A: load(A), B: load(B), "SPY": load("SPY")}).dropna()
rf = pd.read_csv("../data/rf.csv", index_col=0, parse_dates=True)["rf"]
prices.tail()

## Step 2: Excess returns
**What:** subtract the risk-free rate from every daily return.

**Why:** a T-bill pays you for doing nothing. In 2023 that was about 5% a year. If your asset also made 5% that year, it earned nothing for the risk it took. Every number this week is measured *above* the risk-free rate.

`rf` is already a daily rate in decimals. The setup notebook did that conversion.

In [ ]:
rets = prices.pct_change().dropna()
rf = rf.reindex(rets.index).ffill()    # line the T-bill dates up with your trading days
excess = rets.sub(rf, axis=0).dropna()
excess.describe()

**Look for:** the `mean` row should be a hair smaller than the mean of `rets` (about 0.0001 a day). If it's way off, check that `rf` is in decimals, not percent.

## Step 3: Sharpe ratio
**What:** average excess return divided by volatility, annualized.

$$\text{Sharpe} = \frac{\text{mean of daily excess return}}{\text{std of daily excess return}} \times \sqrt{252}$$

**Why:** it answers "how much return did I get for each unit of risk?" The real question isn't whether a Sharpe is high, it's whether it beats just buying SPY. A single asset above 1 over ten years is unusual, so if you get that, check your code before you celebrate.

`excess.mean()` and `excess.std()` work on all three columns at once.

In [ ]:
sharpe = ___ / ___ * np.sqrt(252) #put in the code for "____"
sharpe

## Step 4: Sortino ratio
**What:** same as Sharpe, but the bottom only counts the down days.

**Why:** Sharpe treats a huge up day as "risk." Nobody minds those. Sortino only penalizes losses.

`.clip(upper=0)` turns every positive day into 0, so only the losses are left.

In [ ]:
downside = excess.clip(upper=0)
sortino = excess.mean() / np.sqrt((downside**2).mean()) * np.sqrt(252)
sortino

**Look for:** Sortino will always be higher than Sharpe. For an asset whose ups and downs are symmetric, it's about **1.4×** the Sharpe. Well above 1.4× means the asset's big moves skew up. Below 1.4× means the losses are the lopsided part.

## Step 5: See the regression before you run it
Plot each day as a dot: SPY's excess return on the x-axis, your asset's on the y-axis. Both plots share axes so the slopes are comparable.

- **Slope of the cloud = beta.** Steep means the asset amplifies market moves. Flat means it mostly ignores them.
- **How tight the cloud is = R².** A thin line means the market explains most of the moves. A round blob means it explains little.
- **Where the cloud crosses x = 0 = alpha.** It will look like zero. That's normal: daily alpha is tiny.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, t in zip(axes, [A, B]):
    ax.scatter(excess["SPY"], excess[t], s=2, alpha=0.3)
    ax.set(title=t, xlabel="SPY excess return", ylabel=f"{t} excess return")

Before running the regression, guess: which asset has the higher beta, and which has the higher R²?

**My guess:** [FILL IN]

## Step 6: The market regression (CAPM)
Now put numbers on that picture:

$$r_{\text{asset}} - r_f = \alpha + \beta\,(r_{\text{SPY}} - r_f) + \varepsilon$$

| Piece | What it means |
|---|---|
| **β** (beta) | When SPY moves 1% (above T-bills), the asset moves β% on average |
| **α** (alpha) | The average daily return left over once the market is accounted for. This is "edge" |
| **ε** | Everything else: company news, sector moves, noise |
| **R²** | Share of the asset's daily variance that the market explains |

`sm.add_constant` adds a column of 1s so the regression can estimate α. Leave it out and you're forcing α = 0.

In [ ]:
X = sm.add_constant(excess["SPY"])
fit_A = sm.OLS(excess[A], X).fit()
fit_A.summary()

**Reading the output.** You only need four numbers:

| Where to look | What it is |
|---|---|
| `const` row, `coef` column | Daily alpha. Multiply by 252 to annualize |
| `SPY` row, `coef` column | Beta |
| `const` row, `t` column | Is alpha real? Rule of thumb: \|t\| above 2 means probably not luck. Below 2, you can't claim alpha |
| `R-squared` (top right) | How much of the asset the market explains |

In finance, an R² of 0.3 is not "bad." It just means 70% of the daily moves are about something other than the market.

Now run the same regression for asset B. `X` stays the same.

In [ ]:
fit_B = ___
fit_B.summary()

## Step 7: Check your work
Two quick checks that catch most bugs. Both should come out exactly right. If not, something above is broken.

**1.** Regress SPY on itself. Beta has to be 1 and alpha 0.

In [ ]:
sm.OLS(excess["SPY"], X).fit().params

**2.** Beta is just covariance divided by variance. These two numbers should match.

In [ ]:
excess[A].cov(excess["SPY"]) / excess["SPY"].var(), fit_A.params["SPY"]

## Visual
One beta for ten years hides a lot. A **252-day rolling beta** redoes that cov / var calculation on each trailing year, so you can see market sensitivity change over time. Fill in asset B the same way as A.

The first 251 days are blank. That's normal: there isn't a full year of data yet.

**Look for:**
- Does beta jump in a crisis (early 2020, 2022)? Diversification tends to disappear right when you want it.
- Do the two lines ever cross? That's when your "riskier" asset temporarily wasn't.

In [ ]:
window = 252
roll_beta = pd.DataFrame({
    A: excess[A].rolling(window).cov(excess["SPY"]) / excess["SPY"].rolling(window).var(),
    B: ___,
})
roll_beta.plot(title="252-day rolling beta vs SPY", figsize=(10, 4))
plt.axhline(1, color="gray", linestyle="--");   # beta = 1: moves with the market

## Table
Everything side by side, with SPY as the yardstick. SPY's beta is 1 and its alpha is 0 by definition.

In [ ]:
table = pd.DataFrame({
    A: [sharpe[A], sortino[A], fit_A.params["SPY"], fit_A.params["const"] * 252, fit_A.tvalues["const"], fit_A.rsquared],
    B: [sharpe[B], sortino[B], fit_B.params["SPY"], fit_B.params["const"] * 252, fit_B.tvalues["const"], fit_B.rsquared],
    "SPY": [sharpe["SPY"], sortino["SPY"], 1, 0, np.nan, 1],
}, index=["Sharpe", "Sortino", "Beta", "Alpha (annual)", "Alpha t-stat", "R²"])
table.round(2)

## The Paragraph
> **Once you adjust for risk, which asset is the better deal? How much of each one is just the market, and is there any real alpha left? (Say no if there isn't.)**

Hit these:
- Does the Sharpe ranking match the raw-return ranking from Week 1? If not, why?
- Beta and R²: which asset is mostly riding the market, and which moves on its own?
- Alpha: give the annual number and its t-stat. If \|t\| is under 2, say there's no evidence of alpha.
- Rolling beta: is each asset's market exposure stable, or does it shift in a crisis?

[FILL IN: one paragraph]

---
**Done?** Restart the kernel and **Run All** to check everything runs top to bottom, then commit
(navigate into your project folder in the terminal first):

```bash
git add .
git commit -m "Week 7: risk-adjusted returns"
git push
```